# Object-Oriented Programming



## `class` — the basic shape

A class is declared with `class`, takes an optional parent (defaulting to `object`), and runs its indented body. The shape:

```python
class Customer:
    def __init__(self, name, city):
        self.name = name
        self.city = city

    def greet(self):
        return f"hello, {self.name} from {self.city}"
```

Two things to internalize coming from Java or C++:

- **`self` is explicit and is the first parameter of every instance method.** It is not a keyword — you could name it something else (don't). The interpreter passes the instance as the first argument when you call `obj.method(...)`.
- **`__init__` is the constructor.** There is no `new`. Instances are created by calling the class — `Customer("alice", "delhi")` — which calls `__init__`.

In [ ]:
class Customer:
    def __init__(self, name, city):
        self.name = name
        self.city = city

    def greet(self):
        return f"hello, {self.name} from {self.city}"

c = Customer("alice", "delhi")
print(c.greet())          # hello, alice from delhi
print(c.name)             # alice — attributes are public by default

# Attributes can be set or modified from outside
c.city = "mumbai"
print(c.greet())

# And even added on the fly (sometimes a feature, often a footgun)
c.note = "VIP"
print(c.note)

## Instance attributes vs class attributes

Two different scopes within a class:

- **Class attributes** — defined at the class level (outside any method). Shared by **all instances**. Use for constants, defaults, and class-level state.
- **Instance attributes** — assigned to `self.<name>` inside `__init__` (or any method). Each instance has its own.

The gotcha: lookups go *instance first, then class*. Reading `self.x` finds an instance attribute first, falls back to the class attribute. Writing `self.x = ...` always creates or updates the instance attribute, never the class one. This is fine until you start with a **mutable** class attribute — then all instances share it.

In [ ]:
class Counter:
    total_count = 0       # class attribute — shared across instances

    def __init__(self, name):
        self.name = name  # instance attribute — per instance
        self.count = 0    # instance attribute — per instance

    def tick(self):
        self.count += 1
        Counter.total_count += 1   # write the class attribute via the class

a = Counter("a")
b = Counter("b")
a.tick(); a.tick(); b.tick()

print(a.count, b.count)            # 2 1 — per instance
print(Counter.total_count)         # 3 — shared

# The classic mutable-class-attribute trap
class Buggy:
    items = []        # SHARED across all instances
    def add(self, x):
        self.items.append(x)

x = Buggy(); x.add(1)
y = Buggy(); y.add(2)
print(x.items)        # [1, 2] — surprise, they share the list
print(y.items)        # [1, 2]

## `@classmethod` and `@staticmethod`

Three kinds of methods on a class:

| Kind | First arg | Use |
|---|---|---|
| Instance method | `self` | Operates on one instance |
| `@classmethod` | `cls` | Operates on the class itself — factories, alternative constructors |
| `@staticmethod` | none | Logically grouped with the class, but uses no instance or class state |

`@classmethod` is the Python idiom for **alternative constructors**. The standard library uses it constantly: `dict.fromkeys`, `datetime.fromtimestamp`, `Path.cwd`.

In [ ]:
from datetime import date

class Person:
    def __init__(self, name, year_of_birth):
        self.name = name
        self.year_of_birth = year_of_birth

    def age(self):
        return date.today().year - self.year_of_birth

    @classmethod
    def from_age(cls, name, age):
        """Alternative constructor — build from current age."""
        return cls(name, date.today().year - age)

    @staticmethod
    def is_adult(age):
        """Logically related, but doesn't touch instance or class state."""
        return age >= 18

# Instance method
p = Person("ganesh", 1995)
print(p.age())

# Class method as alternative constructor
q = Person.from_age("alice", 30)
print(q.year_of_birth)

# Static method
print(Person.is_adult(17))   # False

## Visibility — convention, not enforcement

Python has **no real `private` or `protected`**. Two conventions stand in:

- **`_single_leading_underscore`** — by convention, "internal use only." Tools and humans should treat it as private. Nothing stops you from using it; it's a warning sign.
- **`__double_leading_underscore`** — *name mangling*. The interpreter rewrites `self.__x` to `self._ClassName__x`. This avoids accidental collision with subclass attributes; it is **not** a security mechanism.

The honest rule: prefix internal attributes with `_`. Reach for `__` only when you need to avoid subclass name collisions in a class meant to be inherited from. For data hiding from external code, type hints, documentation, and `@property` (next section) do more than naming does.

In [ ]:
class Account:
    def __init__(self, balance):
        self._balance = balance     # convention: private
        self.__secret = "shh"       # name-mangled to _Account__secret

    def deposit(self, amount):
        self._balance += amount

    def get_balance(self):
        return self._balance

acc = Account(100)
print(acc._balance)             # 100 — accessible, "private" is just convention

# Name mangling
print(dir(acc)[:5])             # _Account__secret appears, not __secret
print(acc._Account__secret)     # 'shh' — fully accessible if you know the mangled name
# print(acc.__secret)           # AttributeError

## `@property` — methods that look like attributes

`@property` turns a method into a read-only attribute lookup. Callers write `obj.value` instead of `obj.value()`. The decorator binds a `@property.setter` and `@property.deleter` for the write and delete cases.

The win: you can start with a plain attribute, later replace it with a property when you need validation or computation, and **callers don't have to change**. This is the Python answer to Java's getter/setter ceremony — write a plain attribute first, promote it to a property if and when you need to.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius     # uses the setter below

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError(f"absolute zero is -273.15C, got {value}")
        self._celsius = value

    @property
    def fahrenheit(self):
        """Computed on the fly — no setter, so it's read-only."""
        return self._celsius * 9 / 5 + 32

t = Temperature(25)
print(t.celsius)                # 25 — looks like attribute access
print(t.fahrenheit)             # 77.0 — also looks like attribute access

t.celsius = 30                  # invokes the setter
print(t.fahrenheit)             # 86.0

try:
    t.celsius = -300            # fails validation
except ValueError as e:
    print(f"error: {e}")

## `@dataclass` — boilerplate-free data records

For classes whose main job is to hold data, `@dataclass` (since Python 3.7) generates the boilerplate for you — `__init__`, `__repr__`, `__eq__`, and optionally `__hash__` — based on type annotations.

```python
from dataclasses import dataclass

@dataclass
class Customer:
    name: str
    city: str
    credit_score: int = 700        # defaults supported
```

You get a constructor that takes all the fields, equality by structure, and a useful `repr`. No `self.x = x` boilerplate.

Three flags worth knowing:

- **`@dataclass(frozen=True)`** — instances are immutable; assigning to a field raises. Also makes the class hashable, so instances can be used as dict keys or set members.
- **`@dataclass(slots=True)`** (3.10+) — adds `__slots__`, saving memory and speeding up attribute access.
- **`@dataclass(kw_only=True)`** (3.10+) — constructor accepts only keyword arguments. Good for classes with many fields.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Customer:
    name: str
    city: str
    credit_score: int = 700

c1 = Customer("ganesh", "bengaluru", 780)
c2 = Customer("ganesh", "bengaluru", 780)
c3 = Customer("alice", "delhi")

print(c1)             # Customer(name='ganesh', city='bengaluru', credit_score=780)
print(c1 == c2)       # True — structural equality, generated for free
print(c1 == c3)       # False

# Mutable default — same trap as functions! Use field(default_factory=...)
@dataclass
class Inbox:
    owner: str
    messages: list[str] = field(default_factory=list)   # CORRECT
    # messages: list[str] = []   # WRONG — would raise ValueError

i1 = Inbox("alice")
i1.messages.append("hello")
i2 = Inbox("bob")
print(i1.messages)    # ['hello']
print(i2.messages)    # [] — fresh per instance

# Frozen + hashable
@dataclass(frozen=True)
class Point:
    x: int
    y: int

p = Point(1, 2)
# p.x = 99            # FrozenInstanceError
print(hash(p))        # works — frozen dataclasses are hashable
print({Point(1, 2), Point(1, 2)})   # {Point(x=1, y=2)} — deduplicated

## `NamedTuple` — the typed tuple alternative

`typing.NamedTuple` produces an *immutable* record that's also a real tuple. Use it when you want immutability and don't need anything beyond plain data — slightly less ceremony than `@dataclass(frozen=True)`, and it's a tuple, so it supports indexing and unpacking.

In [ ]:
from typing import NamedTuple

class Point(NamedTuple):
    x: int
    y: int

p = Point(3, 4)
print(p.x, p.y)         # named access
print(p[0], p[1])       # also indexable — it IS a tuple
print(tuple(p))         # (3, 4)

x, y = p                # unpacks like any tuple
print(x, y)

## Inheritance — `class Sub(Base):` and `super()`

A class can inherit from another. The subclass gets all the parent's attributes and methods; you override by defining a same-named method. Inside an override, call `super().method(...)` to invoke the parent's version.

```python
class Animal:
    def __init__(self, name):
        self.name = name

    def greet(self):
        return f"some animal called {self.name}"

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)     # call the parent's __init__
        self.breed = breed

    def greet(self):
        return f"woof, I'm {self.name} the {self.breed}"
```

`super()` without arguments works as long as you're inside a regular method — Python figures out the class and instance from the call site. In older Python 2 code you'll see `super(Dog, self).method()`; you don't need it in 3.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
    def greet(self):
        return f"some animal called {self.name}"

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)
        self.breed = breed
    def greet(self):
        return f"woof, I'm {self.name} the {self.breed}"

class Puppy(Dog):
    def greet(self):
        return super().greet() + " (puppy)"

print(Animal("generic").greet())
print(Dog("rex", "lab").greet())
print(Puppy("buddy", "poodle").greet())

# isinstance checks the inheritance chain
p = Puppy("buddy", "poodle")
print(isinstance(p, Puppy))     # True
print(isinstance(p, Dog))       # True
print(isinstance(p, Animal))    # True

## Multiple inheritance and the MRO

Python supports multiple inheritance. When several base classes define the same method, lookup follows the **MRO** (Method Resolution Order) — Python's C3 linearization, a deterministic ordering of all ancestor classes.

```python
class A:
    def greet(self): return "A"

class B(A):
    def greet(self): return "B -> " + super().greet()

class C(A):
    def greet(self): return "C -> " + super().greet()

class D(B, C):    # inherits from both B and C
    pass
```

`D().greet()` returns `"B -> C -> A"`. The MRO of `D` is `[D, B, C, A, object]` — Python finds `B`'s implementation first, `B.greet` calls `super().greet()`, which in the MRO goes to `C.greet`, which calls `super().greet()`, which goes to `A.greet`.

The rule for the diamond: every class appears exactly once, in an order that respects each parent's ordering and puts subclasses before their parents. You inspect the MRO with `Cls.__mro__` or `Cls.mro()`.

In [ ]:
class A:
    def greet(self): return "A"

class B(A):
    def greet(self): return "B -> " + super().greet()

class C(A):
    def greet(self): return "C -> " + super().greet()

class D(B, C):
    pass

print(D.__mro__)          # the linearization
print(D().greet())        # "B -> C -> A"

# The MRO is computed once, deterministically, at class creation time

## Mixins — composition via inheritance, done right

A **mixin** is a class designed to be inherited from in combination with others, providing a focused set of methods. Mixins usually don't have `__init__` and don't make sense on their own.

The convention: mixin class names end in `Mixin` (or `able`, like `Comparable`, `Iterable`). The pattern shows up across the standard library and Django.

In [ ]:
class SerializableMixin:
    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if not k.startswith("_")}

class PrintableMixin:
    def show(self):
        for k, v in self.to_dict().items():
            print(f"  {k}: {v}")

class User(SerializableMixin, PrintableMixin):
    def __init__(self, name, email):
        self.name = name
        self.email = email

u = User("ganesh", "ganesh@example.com")
print(u.to_dict())
u.show()

## Abstract base classes

Sometimes you want to declare "any subclass MUST implement these methods, but the base class itself can't be instantiated." That's an **abstract base class** (ABC).

The standard library's `abc` module provides `ABC` and the `@abstractmethod` decorator:

```python
from abc import ABC, abstractmethod

class Storage(ABC):
    @abstractmethod
    def save(self, key: str, value: bytes) -> None: ...

    @abstractmethod
    def load(self, key: str) -> bytes: ...

# Cannot instantiate Storage directly — it has abstract methods
# storage = Storage()  # TypeError

class FileStorage(Storage):
    def save(self, key, value):
        ...
    def load(self, key):
        ...

s = FileStorage()  # OK — concrete
```

ABCs are useful for documenting contracts and for the standard-library shapes (`collections.abc.Iterable`, `Sized`, `Container`). For most everyday use, **protocols** (notebook 08) are the more Pythonic choice — they let you express "anything with these methods" without inheritance.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

    @abstractmethod
    def perimeter(self) -> float: ...

    def describe(self) -> str:
        return f"area={self.area():.2f}, perimeter={self.perimeter():.2f}"

class Circle(Shape):
    def __init__(self, radius): self.r = radius
    def area(self): return 3.14159 * self.r ** 2
    def perimeter(self): return 2 * 3.14159 * self.r

class Rectangle(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h
    def perimeter(self): return 2 * (self.w + self.h)

# Cannot instantiate Shape directly
try:
    Shape()
except TypeError as e:
    print(e)

print(Circle(5).describe())
print(Rectangle(3, 4).describe())

## Dunder methods — the protocol catalog

"Dunder" methods (double underscore: `__init__`, `__repr__`, `__eq__`, etc.) are how you hook into Python's syntax. Define `__add__` and your class works with `+`. Define `__len__` and `len(obj)` works. Define `__iter__` and `for x in obj:` works.

The high-value ones to know:

| Dunder | Triggered by | Use |
|---|---|---|
| `__init__` | `Cls(...)` | constructor |
| `__repr__` | `repr(x)`, REPL | unambiguous debug form |
| `__str__` | `str(x)`, `print(x)` | human-readable form |
| `__eq__`, `__hash__` | `==`, set/dict membership | structural equality |
| `__lt__`, `__le__`, ... | `<`, `<=`, sorted | ordering |
| `__len__` | `len(x)` | size |
| `__getitem__` | `x[i]` | indexing, slicing |
| `__iter__`, `__next__` | `for ... in x` | iteration (notebook 07) |
| `__contains__` | `item in x` | membership |
| `__call__` | `x(...)` | makes instances callable like functions |
| `__enter__`, `__exit__` | `with x:` | context manager (notebook 06) |
| `__bool__` | `bool(x)`, `if x:` | truthiness |
| `__add__`, `__sub__`, ... | arithmetic operators | operator overloading |

In [ ]:
class Money:
    def __init__(self, cents):
        self.cents = cents

    def __repr__(self):
        return f"Money(cents={self.cents})"

    def __str__(self):
        return f"${self.cents // 100}.{self.cents % 100:02d}"

    def __eq__(self, other):
        return isinstance(other, Money) and self.cents == other.cents

    def __hash__(self):
        return hash(("Money", self.cents))

    def __lt__(self, other):
        return self.cents < other.cents

    def __add__(self, other):
        return Money(self.cents + other.cents)

    def __bool__(self):
        return self.cents != 0

m1 = Money(500)
m2 = Money(125)

print(repr(m1))       # Money(cents=500)
print(str(m1))        # $5.00
print(m1 + m2)        # $6.25
print(m1 == Money(500))   # True
print(sorted([m1, m2]))   # ordered
print({m1, Money(500)})   # set works — uses __hash__
print(bool(Money(0)))     # False

## `__slots__` — the memory and speed knob

By default, every instance has a `__dict__` for attribute storage. That's flexible (you can add attributes later) but each instance pays the dict overhead — both memory and a slight indirection on every attribute access.

`__slots__` declares the allowed attributes upfront. The interpreter then uses a fixed-size struct instead of a dict. The win: less memory per instance, faster attribute access. The cost: no dynamic attribute additions; less convenient with multiple inheritance.

Reach for `__slots__` when you'll create *millions* of instances of one class — data points, graph nodes, log records. Most classes don't need it.

In [ ]:
class WithDict:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlots:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x = x
        self.y = y

a = WithDict(1, 2)
b = WithSlots(1, 2)

# Dynamic attributes — works on WithDict, fails on WithSlots
a.z = 3
print(a.z)
try:
    b.z = 3
except AttributeError as e:
    print(f"slots: {e}")

# Dataclasses get slots=True since 3.10
from dataclasses import dataclass
@dataclass(slots=True)
class Point:
    x: int
    y: int

p = Point(3, 4)
print(p)

## `isinstance`, `issubclass`, and the duck-typing alternative

- **`isinstance(x, T)`** — is `x` an instance of `T` (or a subclass)?
- **`issubclass(S, T)`** — is `S` a subclass of `T`?

Both accept a tuple of types for "any of these": `isinstance(x, (int, float))`.

**Duck typing** is the more Pythonic alternative: instead of asking *"is this thing a `File`?"*, just call `.read()` on it. If it works, it's file-enough. The acronym **EAFP** — Easier to Ask Forgiveness than Permission — captures the Pythonic preference: try the operation, catch the exception if it fails. Compare with **LBYL** — Look Before You Leap — typing checks before acting, common in defensive C code.

`isinstance` still earns its keep for type-based dispatch and for `match`/`case` patterns. But before reaching for it, ask: can I just do what I want, and let the exception speak for itself?

## When NOT to write a class

A function is simpler than a class. Reach for a class only when you have a reason. The good reasons:

- The thing has **state** that survives across multiple operations.
- The thing has **multiple operations** that all share that state.
- The thing has **identity** — two with the same fields are not necessarily the same thing.

The wrong reasons (signs you should write a function instead):

- "I want to organize related functions" → use a module.
- "I want to attach configuration to a few methods" → use a closure or `functools.partial`.
- "I want named arguments and a `repr`" → use a `@dataclass`, not a hand-written class.
- "I want one of these" → use a module-level constant or a `@dataclass(frozen=True)` literal.

The Python idiom prefers the simplest tool that does the job. A function is simpler than a closure. A closure is simpler than a class. A `@dataclass` is simpler than a hand-written class with `__init__`. Reach down the ladder only when you need more.